# Serving — agg_video_trending, agg_video_reactions, agg_video_top_comments (dashboard v3, added 2026-11-03)

Page 2 "Video review đang hot" of the dashboard. **Full overwrite** every run (`CREATE OR REPLACE`). Run after `gold/agg_daily_snapshots` (`agg_video_daily`) and `gold/agg_restaurant_ranking` (restaurant names and scores).

**agg_video_trending** — 1 restaurant review video (`dim_video.video_type = 'restaurant_review'`; mukbang, recipes and other `food_other` videos are left out):
- `views_7d` = latest `view_count` − the last daily count on or before 7 days earlier. A video with no count before the window counts all its views when it was **published** in the window, else only the views since its first count (an old viral video found by a search today is not trending).
- `views_per_day` = `views_7d` ÷ days in the window it was online (a 2-day-old video is not beaten by a 7-day-old one just for being older).
- `trending_rank` = rank by `views_7d` among videos with ≥ `TREND_MIN_VIEWS_7D` (1,000), else NULL.
- `restaurants` = the restaurants the video mentions (`fact_video_restaurant_mention`, located ones) with their `restaurant_score`; `restaurants_text` for the table; `city` = the most frequent city of those restaurants.

**agg_video_reactions** — top `TOP_VIDEOS` (30) trending videos × reaction (`silver.stg_comment_reaction`): `comment_count`, `like_count`, `share_of_video`. Labels in Vietnamese.

**agg_video_top_comments** — top 30 trending videos × their 20 most liked comments, with the reaction (NULL = not labeled yet).

**Vietnam only (datle 2026-11-03):** `trending_rank` only for videos with a Vietnamese title (diacritics), a channel with country VN, or a located restaurant (`is_vietnam`); foreign food videos (India, US...) stay in the table without a rank.

**Audience sentiment (datle 2026-11-03):** `comments_analyzed`, `positive_share`, `negative_share` of the video's comments scored by `stg/comment_sentiment` (positive / negative / neutral), else by the reaction of `stg/comment_reaction` (complaint = negative, ask_info / other = neutral, rest = positive).


In [ ]:
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))
dbutils.widgets.text("serving", "prod.serving")
SERVING = dbutils.widgets.get("serving")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SERVING}")

TRENDING = f"{SERVING}.agg_video_trending"
REACTIONS_T = f"{SERVING}.agg_video_reactions"
TOP_COMMENTS_T = f"{SERVING}.agg_video_top_comments"
RANKING = f"{SERVING}.agg_restaurant_ranking"
REACTION_SRC = f"{SILVER}.stg_comment_reaction"   # stg/comment_reaction (may not exist before its first run)
TREND_DAYS = 7
TREND_MIN_VIEWS_7D = 1000
TOP_VIDEOS = 30
TOP_COMMENTS = 20
VI_CHARS = "àáảãạăằắẳẵặâầấẩẫậđèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵĐ"  # Vietnamese title = Vietnamese video
REACTION_LABELS = {"praise_food": "Khen món / quán", "complaint": "Chê", "craving": "Thèm, muốn đi ăn",
                   "ask_info": "Hỏi địa chỉ / giá", "praise_video": "Khen video / reviewer", "other": "Khác"}
reaction_case = "CASE r.reaction " + " ".join(f"WHEN '{k}' THEN '{v}'" for k, v in REACTION_LABELS.items()) + " END"
if not spark.catalog.tableExists(REACTION_SRC):
    REACTION_SRC = "(SELECT CAST(NULL AS STRING) AS comment_id, CAST(NULL AS STRING) AS reaction WHERE false)"
print(f"Serving targets: {TRENDING}, {REACTIONS_T}, {TOP_COMMENTS_T}")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {TRENDING}
    COMMENT 'Serving: 1 restaurant review video with views gained in the last {TREND_DAYS} days, rank and the restaurants it mentions. Full rebuild.'
    AS
    WITH d AS (SELECT video_key, snapshot_date_key, to_date(CAST(snapshot_date_key AS STRING), 'yyyyMMdd') AS day,
                      view_count, like_count, comment_count
               FROM {SERVING}.agg_video_daily WHERE view_count IS NOT NULL),
    w AS (SELECT MAX(day) AS today, date_sub(MAX(day), {TREND_DAYS}) AS start FROM d),
    latest AS (SELECT video_key, max_by(view_count, day) AS view_count, max_by(like_count, day) AS like_count,
                      max_by(comment_count, day) AS comment_count, min_by(view_count, day) AS first_views
               FROM d GROUP BY video_key),
    base AS (SELECT d.video_key, max_by(d.view_count, d.day) AS views_then
             FROM d CROSS JOIN w WHERE d.day <= w.start GROUP BY d.video_key),
    v AS (
        SELECT dv.video_key, dv.video_id, dv.title, dv.video_url, dv.thumbnail_url, dv.channel_id,
               ch.channel_title, ch.country AS channel_country, to_date(dv.published_at) AS published_date,
               l.view_count, l.like_count, l.comment_count,
               greatest(l.view_count - CASE WHEN b.views_then IS NOT NULL THEN b.views_then
                                            WHEN to_date(dv.published_at) > w.start THEN 0
                                            ELSE l.first_views END, 0) AS views_7d,
               least(greatest(datediff(w.today, to_date(dv.published_at)), 1), {TREND_DAYS}) AS days_in_window
        FROM latest l
        CROSS JOIN w
        JOIN {GOLD}.dim_video dv ON dv.video_key = l.video_key AND dv.video_type = 'restaurant_review'
        LEFT JOIN {GOLD}.dim_channel ch ON ch.channel_id = dv.channel_id
        LEFT JOIN base b ON b.video_key = l.video_key
    ),
    rest AS (   -- located restaurants the video mentions, best score first
        SELECT m.video_key,
               array_sort(collect_list(named_struct('neg_score', -coalesce(r.restaurant_score, -1),
                                                    'restaurant_id', r.restaurant_id, 'restaurant_name', r.restaurant_name,
                                                    'restaurant_score', r.restaurant_score, 'cuisine', r.cuisine,
                                                    'district', r.district, 'city', r.city))) AS restaurants,
               mode(r.city) AS city
        FROM {GOLD}.fact_video_restaurant_mention m
        JOIN {RANKING} r ON r.restaurant_id = m.restaurant_id
        WHERE m.restaurant_id <> 'unknown'
        GROUP BY m.video_key
    )
    , senti AS (   -- audience sentiment of the analyzed comments (stg/comment_sentiment, ai_query)
        SELECT c.video_key, COUNT(*) AS comments_analyzed,
               COUNT_IF(s.sentiment = 'positive') AS positive_comments,
               COUNT_IF(s.sentiment = 'negative') AS negative_comments
        FROM {GOLD}.dim_comment c
        JOIN (SELECT coalesce(s.comment_id, r.comment_id) AS comment_id,
                     coalesce(s.sentiment, CASE r.reaction WHEN 'complaint' THEN 'negative'
                                                           WHEN 'ask_info' THEN 'neutral' WHEN 'other' THEN 'neutral'
                                                           ELSE 'positive' END) AS sentiment
              FROM {SILVER}.stg_comment_sentiment s
              FULL OUTER JOIN {REACTION_SRC} r ON r.comment_id = s.comment_id) s ON s.comment_id = c.comment_id
        WHERE c.comment_key <> -1
        GROUP BY c.video_key
    )
    , x AS (   -- Vietnam: Vietnamese title, a VN channel or a located restaurant (datle 2026-11-03: hide foreign videos)
        SELECT v.*, r.restaurants AS r_restaurants, r.city AS r_city,
               se.comments_analyzed, se.positive_comments, se.negative_comments,
               (v.title RLIKE '[{VI_CHARS}]' OR v.channel_country = 'VN' OR size(r.restaurants) > 0) AS is_vietnam
        FROM v LEFT JOIN rest r ON r.video_key = v.video_key
        LEFT JOIN senti se ON se.video_key = v.video_key
    )
    SELECT v.video_id, v.video_key, v.title, v.video_url, v.thumbnail_url, v.channel_id, v.channel_title,
           v.published_date, v.view_count, v.like_count, v.comment_count,
           round(v.like_count / NULLIF(v.view_count, 0), 4)                 AS like_rate,
           CAST(v.views_7d AS BIGINT)                                       AS views_7d,
           CAST(round(v.views_7d / v.days_in_window) AS BIGINT)             AS views_per_day,
           CASE WHEN v.is_vietnam AND v.views_7d >= {TREND_MIN_VIEWS_7D}
                THEN CAST(RANK() OVER (PARTITION BY v.is_vietnam AND v.views_7d >= {TREND_MIN_VIEWS_7D}
                                       ORDER BY v.views_7d DESC, v.view_count DESC) AS INT) END AS trending_rank,
           transform(v.r_restaurants, x -> named_struct('restaurant_id', x.restaurant_id, 'restaurant_name', x.restaurant_name,
                                                      'restaurant_score', x.restaurant_score, 'cuisine', x.cuisine,
                                                      'district', x.district))      AS restaurants,
           CAST(coalesce(size(v.r_restaurants), 0) AS INT)                     AS restaurant_count,
           array_join(transform(v.r_restaurants, x -> concat(x.restaurant_name,
                      CASE WHEN x.restaurant_score IS NOT NULL THEN concat(' (', CAST(x.restaurant_score AS STRING), ')') ELSE '' END)),
                      ' · ')                                                AS restaurants_text,
           array_join(array_distinct(filter(transform(v.r_restaurants, x -> x.cuisine), c -> c IS NOT NULL AND c <> 'Chưa rõ')), ', ') AS cuisines,
           COALESCE(v.r_city, 'Chưa rõ')                                      AS city,
           v.is_vietnam,
           CAST(coalesce(v.comments_analyzed, 0) AS INT)                     AS comments_analyzed,
           round(v.positive_comments / NULLIF(v.comments_analyzed, 0), 4)    AS positive_share,
           round(v.negative_comments / NULLIF(v.comments_analyzed, 0), 4)    AS negative_share,
           current_timestamp()                                              AS etl_date
    FROM x v
""")
display(spark.sql(f"""SELECT trending_rank, title, channel_title, published_date, views_7d, views_per_day, restaurants_text
                      FROM {TRENDING} WHERE trending_rank <= 15 ORDER BY trending_rank"""))

In [ ]:
# --- Comments of the top trending videos: reaction mix + the most liked comments ---
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW top_video_comments AS
    SELECT t.video_id, t.video_key, t.title, t.trending_rank, c.comment_id, c.comment_text, c.like_count, c.reply_count,
           CAST(c.posted_at AS DATE) AS posted_date, r.reaction, {reaction_case} AS reaction_label,
           ROW_NUMBER() OVER (PARTITION BY t.video_id ORDER BY c.like_count DESC NULLS LAST, c.comment_id) AS like_rank
    FROM {TRENDING} t
    JOIN {GOLD}.dim_comment c ON c.video_key = t.video_key AND c.comment_key <> -1
    LEFT JOIN {REACTION_SRC} r ON r.comment_id = c.comment_id
    WHERE t.trending_rank <= {TOP_VIDEOS}
""")
spark.sql(f"""
    CREATE OR REPLACE TABLE {REACTIONS_T}
    COMMENT 'Serving: top {TOP_VIDEOS} trending videos x audience reaction of their labeled comments (stg_comment_reaction). Full rebuild.'
    AS
    SELECT video_id, title, trending_rank, reaction, reaction_label,
           CAST(COUNT(*) AS INT)                                           AS comment_count,
           CAST(SUM(coalesce(like_count, 0)) AS BIGINT)                    AS like_count,
           round(COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY video_id), 4) AS share_of_video,
           current_timestamp()                                             AS etl_date
    FROM top_video_comments
    WHERE reaction IS NOT NULL
    GROUP BY video_id, title, trending_rank, reaction, reaction_label
""")
spark.sql(f"""
    CREATE OR REPLACE TABLE {TOP_COMMENTS_T}
    COMMENT 'Serving: top {TOP_VIDEOS} trending videos x their {TOP_COMMENTS} most liked comments with the reaction (NULL = not labeled yet). Full rebuild.'
    AS
    SELECT video_id, title, trending_rank, comment_id, comment_text, like_count, reply_count, posted_date,
           reaction, coalesce(reaction_label, 'Chưa phân loại') AS reaction_label, like_rank,
           current_timestamp() AS etl_date
    FROM top_video_comments
    WHERE like_rank <= {TOP_COMMENTS}
""")
s = spark.sql(f"""SELECT (SELECT COUNT(*) FROM {TRENDING}) AS videos, (SELECT COUNT(*) FROM {TRENDING} WHERE trending_rank IS NOT NULL) AS trending,
                         (SELECT COUNT(DISTINCT video_id) FROM {TRENDING}) AS grain,
                         (SELECT SUM(comment_count) FROM {REACTIONS_T}) AS labeled,
                         (SELECT COUNT(*) FROM {TOP_COMMENTS_T}) AS top_comments""").collect()[0]
print(f"📊 {TRENDING}: {s['videos']} restaurant review videos, {s['trending']} trending · {REACTIONS_T}: {s['labeled']} labeled comments · "
      f"{TOP_COMMENTS_T}: {s['top_comments']} rows " + ("✅" if s["videos"] == s["grain"] else "❌ duplicate video_id"))
assert s["videos"] == s["grain"], "duplicate video_id in agg_video_trending"
display(spark.sql(f"SELECT reaction_label, SUM(comment_count) AS n FROM {REACTIONS_T} GROUP BY 1 ORDER BY 2 DESC"))